# Step 1: Data Collection

## Pre-requisites

Install required packages

In [1]:
%pip install -r "..\\requirements.txt"

Note: you may need to restart the kernel to use updated packages.


Import required packages

In [2]:
import pandas as pd
import numpy as np
import requests
import time
import os, json, asyncio, logging
from datetime import date, timedelta
from pathlib import Path
import aiohttp, aiofiles
from tqdm.asyncio import tqdm

On the OS level (Windows, Linux, Bash, or conda), create the following environment variables to store the respective API keys:
- EBIRD_API_KEY
- NCEI_API_KEY
- CENSUS_API_KEY

Alternatively, for each `api_key` or `api_token` parameter, replace `"YOUR_API_KEY_HERE"` with the corresponding API key.

## Get eBird data on bird sightings

### Get historic observations

In [3]:
'''
Historic observations data is retrieved from the eBird API service for a specific date.
We make repeated API requests to the eBird API service to get data for each date from 2021-01-01 to 2025-12-31.
Due to the large number of requests, we store the downloaded data in a cache such that if the download is interrupted, it can pick up from where it had left off the next time.
The code adds a short delay to the next request if the API service deems that the rate limit has been reached.
The async library is used to run multiple tasks asynchronously so as to improve performance and reduce execution time.
'''

# Configuration

API_KEY        = os.getenv("EBIRD_API_KEY", "YOUR_API_KEY_HERE")
REGION         = "US-CO"
START_DATE     = date(2021, 1, 1)
END_DATE       = date(2025, 12, 31)
OUTPUT_CSV     = "..\data\ebird_co.csv"
CACHE_DIR      = Path("..\data\ebird_cache")
MAX_CONCURRENT = 10

PARAMS = {
    "rank": "create", "maxResults": 10000,
    "includeProvisional": "true", "hotspot": "false", "detail": "full",
}

# Fetch logic

def all_dates(start, end):
    d, out = start, []
    while d <= end:
        out.append(d); d += timedelta(days=1)
    return out

async def load_cache(d):
    p = CACHE_DIR / f"{d}.json"
    if p.exists():
        async with aiofiles.open(p) as f:
            return json.loads(await f.read())

async def save_cache(d, records):
    CACHE_DIR.mkdir(parents=True, exist_ok=True)
    async with aiofiles.open(CACHE_DIR / f"{d}.json", "w") as f:
        await f.write(json.dumps(records))

async def fetch_date(session, sem, d):
    cached = await load_cache(d)
    if cached is not None:
        return cached
    url = f"https://api.ebird.org/v2/data/obs/{REGION}/historic/{d.year}/{d.month:02d}/{d.day:02d}"
    async with sem:
        for attempt in range(1, 5):
            try:
                async with session.get(url, params=PARAMS, timeout=aiohttp.ClientTimeout(total=30)) as r:
                    if r.status == 200:
                        records = await r.json(content_type=None)
                        await save_cache(d, records); return records
                    elif r.status == 404:
                        await save_cache(d, []); return []
                    elif r.status == 429:
                        await asyncio.sleep(3 * 2**attempt)
                    else:
                        await asyncio.sleep(3 * attempt)
            except Exception:
                await asyncio.sleep(3 * attempt)
    return []

async def run():
    dates = all_dates(START_DATE, END_DATE)
    print(f"Fetching {len(dates)} days for {REGION}  ({START_DATE} → {END_DATE})")
    sem = asyncio.Semaphore(MAX_CONCURRENT)
    headers = {"X-eBirdApiToken": API_KEY}
    async with aiohttp.ClientSession(headers=headers,
                                     connector=aiohttp.TCPConnector(limit=MAX_CONCURRENT+5)) as session:
        tasks = [fetch_date(session, sem, d) for d in dates]
        results = await tqdm.gather(*tasks, desc="Fetching", unit="day")

    all_records = []
    for d, recs in zip(dates, results):
        for r in recs: r["queryDate"] = str(d)
        all_records.extend(recs)

    print(f"\nTotal records: {len(all_records):,}")
    if not all_records:
        return None

    df = pd.DataFrame(all_records)
    df.columns = [c[0].lower() + c[1:] for c in df.columns]
    priority = ["speciesCode","comName","sciName","obsDt","queryDate",
                "howMany","lat","lng","locId","locName","obsValid","subId"]
    cols = [c for c in priority if c in df.columns] + [c for c in df.columns if c not in priority]
    df = df[cols]
    df["howMany"] = pd.to_numeric(df.get("howMany"), errors="coerce")
    df["obsDt"]   = pd.to_datetime(df.get("obsDt"),  errors="coerce")
    df.sort_values(["obsDt","comName"], inplace=True, ignore_index=True)
    df.to_csv(OUTPUT_CSV, index=False)
    print(f"Species : {df['comName'].nunique():,}")
    print(f"Saved   : {OUTPUT_CSV}")
    return df

# Run  (Jupyter has a running event loop, so we use await directly)

ebird_df = await run()

<>:15: SyntaxWarning: "\d" is an invalid escape sequence. Such sequences will not work in the future. Did you mean "\\d"? A raw string is also an option.
<>:16: SyntaxWarning: "\d" is an invalid escape sequence. Such sequences will not work in the future. Did you mean "\\d"? A raw string is also an option.
C:\Users\kenny\AppData\Local\Temp\ipykernel_74832\1944291899.py:15: SyntaxWarning: "\d" is an invalid escape sequence. Such sequences will not work in the future. Did you mean "\\d"? A raw string is also an option.
  OUTPUT_CSV     = "..\data\ebird_co.csv"
C:\Users\kenny\AppData\Local\Temp\ipykernel_74832\1944291899.py:16: SyntaxWarning: "\d" is an invalid escape sequence. Such sequences will not work in the future. Did you mean "\\d"? A raw string is also an option.
  CACHE_DIR      = Path("..\data\ebird_cache")


Fetching 1826 days for US-CO  (2021-01-01 → 2025-12-31)


Fetching: 100%|██████████| 1826/1826 [10:15<00:00,  2.97day/s]



Total records: 354,753
Species : 567
Saved   : ..\data\ebird_co.csv


### Get eBird taxonomy information

In [4]:
url = "https://api.ebird.org/v2/ref/taxonomy/ebird"
headers = {"X-eBirdApiToken": API_KEY}
params = {
    "fmt": "json"
}
ebird_taxonomy_response_json = requests.get(url, headers = headers, params = params).json()
ebird_taxonomy_df = pd.DataFrame(ebird_taxonomy_response_json)
ebird_taxonomy_df.to_csv("..\\data\\ebird_taxonomy.csv")

<>:8: SyntaxWarning: "\d" is an invalid escape sequence. Such sequences will not work in the future. Did you mean "\\d"? A raw string is also an option.
<>:8: SyntaxWarning: "\d" is an invalid escape sequence. Such sequences will not work in the future. Did you mean "\\d"? A raw string is also an option.
C:\Users\kenny\AppData\Local\Temp\ipykernel_74832\3383966064.py:8: SyntaxWarning: "\d" is an invalid escape sequence. Such sequences will not work in the future. Did you mean "\\d"? A raw string is also an option.
  ebird_taxonomy_df.to_csv("..\data\ebird_taxonomy.csv")


## Get NCEI data on climate

### CO Weather Stations

In [5]:
'''
Get all CO weather stations that provide data from 2021-01-01 to 2025-12-31.
'''

base_url = "https://www.ncei.noaa.gov/cdo-web/api/v2/stations"
api_token = os.getenv("NCEI_API_KEY", "YOUR_API_KEY_HERE")
headers = {"token": api_token}
params = {
    "datasetid": "GSOM",
    "locationid": "FIPS:08",
    "startdate": "2021-01-01",
    "enddate": "2025-12-31",
    "limit": 1000,
    "offset": 1
}

all_results = []
fetch_complete = False

while not fetch_complete:
    print(f"Fetching records starting at offset {params['offset']}...")
    
    response = requests.get(base_url, headers=headers, params=params)
    
    if response.status_code == 200:
        data = response.json()
        
        # Add the 'results' from this page to our master list
        if 'results' in data:
            all_results.extend(data['results'])
            
            # Metadata tells us how many total records exist
            total_count = data['metadata']['resultset']['count']
            
            # If we've collected everything, stop the loop
            if len(all_results) >= total_count:
                fetch_complete = True
            else:
                # Increment offset for the next page
                params['offset'] += 1000
                time.sleep(0.2) # Small delay to stay under 5 req/sec
        else:
            print("No more results found.")
            break
            
    elif response.status_code == 429:
        print("Rate limit hit! Sleeping for 5 seconds...")
        time.sleep(5)
    else:
        print(f"Error {response.status_code}: {response.text}")
        break

print(f"Done! Total records collected: {len(all_results)}")

stations_df = pd.DataFrame(all_results)
stations_df.to_csv("..\data\co_weather_stations.csv")

<>:56: SyntaxWarning: "\d" is an invalid escape sequence. Such sequences will not work in the future. Did you mean "\\d"? A raw string is also an option.
<>:56: SyntaxWarning: "\d" is an invalid escape sequence. Such sequences will not work in the future. Did you mean "\\d"? A raw string is also an option.
C:\Users\kenny\AppData\Local\Temp\ipykernel_74832\1241732292.py:56: SyntaxWarning: "\d" is an invalid escape sequence. Such sequences will not work in the future. Did you mean "\\d"? A raw string is also an option.
  stations_df.to_csv("..\data\co_weather_stations.csv")


Fetching records starting at offset 1...
Fetching records starting at offset 1001...
Done! Total records collected: 1763


### GSOM Weather Data

In [6]:
'''
Get weather data from Global Summary of the Month dataset. Dataset ID: GSOM.
Filters:
1. CO's locationid: FIPS:08
2. startdate and enddate - 2021-01-01 to 2025-12-31
3. datatypeid. E.g.: ["TMAX", "TMIN", "PRCP", "AWND", "RHMN", "RHMX"]

API Limitations:
1. Max 1000 records per response
Solution: When there are more than 1000 records to fetch, update the "offset" parameter to the next record to fetch. Then send another request to the API endpoint to retrieve the next batch of records
2. 5 requests per second, 10,000 requests per day
Solution: Code checks for error 429: Too many requests. If error 429 is received, use time.sleep(5) to wait for 5 seconds before re-sending the failed request
3. Request for daily data is limited to a one-year range
Solution: Repeat the API calls 5 times, once for each year
4. Request for monthly and annual data is limited to a ten-year range

'''
def get_ncei_data():

    dataset = []
    
    def get_ncei_datatype(data_type):

        base_url = "https://www.ncei.noaa.gov/cdo-web/api/v2/data"
        api_token = os.getenv("NCEI_API_KEY", "YOUR_API_KEY_HERE")
        headers = {"token": api_token}
        params = {
            "datasetid": "GSOM",
            "locationid": "FIPS:08",
            "startdate": "2021-01-01",
            "enddate": "2025-12-31",
            "datatypeid": data_type,
            "limit": 1000,
            "offset": 1
        }

        all_results = []
        fetch_complete = False

        while not fetch_complete:
            print(f"Fetching records starting at offset {params['offset']}...")
            
            response = requests.get(base_url, headers=headers, params=params)
            
            if response.status_code == 200:
                data = response.json()
                
                # Add the 'results' from this page to our master list
                if 'results' in data:
                    all_results.extend(data['results'])
                    
                    # Metadata tells us how many total records exist
                    total_count = data['metadata']['resultset']['count']
                    
                    # If we've collected everything, stop the loop
                    if len(all_results) >= total_count:
                        fetch_complete = True
                    else:
                        # Increment offset for the next page
                        params['offset'] += 1000
                        time.sleep(0.2) # Small delay to stay under 5 req/sec
                else:
                    print("No more results found.")
                    break
                    
            elif response.status_code == 429:
                print("Rate limit hit! Sleeping for 5 seconds...")
                time.sleep(5)
            elif response.status_code == 503:
                print("Service unavailable (503)! Retrying in 5 seconds...")
                time.sleep(5)
            else:
                print(f"Error {response.status_code}: {response.text}")
                break

        print(f"Done! Total records collected for {data_type}: {len(all_results)}")

        return(all_results)
    
    data_types = ["TMAX", "TMIN", "PRCP", "AWND", "RHMN", "RHMX"]
    
    for data_type in data_types:
        dataset.extend(get_ncei_datatype(data_type))

    return(dataset)

weather_df = pd.DataFrame(get_ncei_data())
weather_df.to_csv("..\data\co_weather.csv")

Fetching records starting at offset 1...


<>:88: SyntaxWarning: "\d" is an invalid escape sequence. Such sequences will not work in the future. Did you mean "\\d"? A raw string is also an option.
<>:88: SyntaxWarning: "\d" is an invalid escape sequence. Such sequences will not work in the future. Did you mean "\\d"? A raw string is also an option.
C:\Users\kenny\AppData\Local\Temp\ipykernel_74832\2355937146.py:88: SyntaxWarning: "\d" is an invalid escape sequence. Such sequences will not work in the future. Did you mean "\\d"? A raw string is also an option.
  weather_df.to_csv("..\data\co_weather.csv")


Fetching records starting at offset 1001...
Fetching records starting at offset 2001...
Fetching records starting at offset 3001...
Fetching records starting at offset 4001...
Fetching records starting at offset 5001...
Fetching records starting at offset 6001...
Fetching records starting at offset 7001...
Fetching records starting at offset 8001...
Fetching records starting at offset 9001...
Fetching records starting at offset 10001...
Fetching records starting at offset 11001...
Fetching records starting at offset 12001...
Fetching records starting at offset 13001...
Fetching records starting at offset 14001...
Fetching records starting at offset 15001...
Fetching records starting at offset 16001...
Fetching records starting at offset 17001...
Fetching records starting at offset 18001...
Fetching records starting at offset 19001...
Fetching records starting at offset 20001...
Fetching records starting at offset 21001...
Done! Total records collected for TMAX: 21181
Fetching records s

## Get FEMA data on natural disasters

In [7]:
'''
FEMA natural disaster declarations from 2021-01-01 to 2025-12-31.
'''

base_url = "https://www.fema.gov/api/open/v2/DisasterDeclarationsSummaries"
params = {
    "$allrecords": "true",
    "$filter": "state eq 'CO' and declarationDate ge '2021-01-01' and declarationDate le '2025-12-31'"
}

response = requests.get(base_url, params=params)
natural_disasters_df = pd.DataFrame(response.json()["DisasterDeclarationsSummaries"])
natural_disasters_df.to_csv("..\data\co_natural_disasters.csv")

<>:13: SyntaxWarning: "\d" is an invalid escape sequence. Such sequences will not work in the future. Did you mean "\\d"? A raw string is also an option.
<>:13: SyntaxWarning: "\d" is an invalid escape sequence. Such sequences will not work in the future. Did you mean "\\d"? A raw string is also an option.
C:\Users\kenny\AppData\Local\Temp\ipykernel_74832\1886168083.py:13: SyntaxWarning: "\d" is an invalid escape sequence. Such sequences will not work in the future. Did you mean "\\d"? A raw string is also an option.
  natural_disasters_df.to_csv("..\data\co_natural_disasters.csv")


## Get Census Bureau data on urban population

In [8]:
'''
Get CO urban population data from 2020, the latest available from Census Bureau
'''

def get_colorado_urban_data():
    # Base URL for Decennial Census 2020 (contains explicit Urban/Rural variables)
    # Variable P2_001N: Total Population
    # Variable P2_002N: Urban Population
    base_url = "https://api.census.gov/data/2020/dec/dhc"

    api_key = os.getenv("CENSUS_API_KEY", "YOUR_API_KEY_HERE")

    params = {
        "get": "NAME,P2_001N,P2_002N",
        "for": "county:*",
        "in": "state:08", # 08 is Colorado's FIPS code
    }

    if api_key:
        params["key"] = api_key

    response = requests.get(base_url, params=params)

    if response.status_code == 200:
        data = response.json()
        # Convert to DataFrame: first row is headers
        df = pd.DataFrame(data[1:], columns=data[0])

        # Rename columns for clarity
        df.columns = ['County Name', 'Total Pop', 'Urban Pop', 'State FIPS', 'County FIPS']

        # Convert numeric columns
        df['Total Pop'] = pd.to_numeric(df['Total Pop'])
        df['Urban Pop'] = pd.to_numeric(df['Urban Pop'])
        df['Urban Percentage'] = (df['Urban Pop'] / df['Total Pop'] * 100).round(2)

        return df
    else:
        print(f"Error: {response.status_code}")
        print(response.text)
        return None

co_urban_df = get_colorado_urban_data()

co_urban_df.to_csv("..\data\co_urban_pop.csv")

<>:45: SyntaxWarning: "\d" is an invalid escape sequence. Such sequences will not work in the future. Did you mean "\\d"? A raw string is also an option.
<>:45: SyntaxWarning: "\d" is an invalid escape sequence. Such sequences will not work in the future. Did you mean "\\d"? A raw string is also an option.
C:\Users\kenny\AppData\Local\Temp\ipykernel_74832\3640580417.py:45: SyntaxWarning: "\d" is an invalid escape sequence. Such sequences will not work in the future. Did you mean "\\d"? A raw string is also an option.
  co_urban_df.to_csv("..\data\co_urban_pop.csv")
